In [1]:
import ray
import time
import numpy as np
from numpy import random

In [2]:
ray.init()

2026-05-27 14:49:28,756	WARNING services.py:2213 -- WARNING: The object store is using /tmp/ray instead of /dev/shm because /dev/shm has only 2147483648 bytes available. This will harm performance! You may be able to free up space by deleting files in /dev/shm. If you are inside a Docker container, you can increase /dev/shm size by passing '--shm-size=10.13gb' to 'docker run' (or add it to the run_options list in a Ray cluster config). Make sure to set this to more than 30% of available RAM.
2026-05-27 14:49:29,867	INFO worker.py:2003 -- Started a local Ray instance. View the dashboard at 127.0.0.1:8265 
/home/ray/anaconda3/lib/python3.10/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


Python version:,3.10.20
Ray version:,2.55.1
Dashboard:,http://127.0.0.1:8265


### DataNode

In [3]:
@ray.remote
class DataNode:
    def __init__(self, max_chunks):
        self.chunks = [None for _ in range(max_chunks)]
        self.num_chunks = 0

    # Add new chunk and return its index
    def add_chunk(self, data):
        for index, chunk in enumerate(self.chunks):
            if chunk is None:
                self.chunks[index] = data
                return index

    # Update chunk at given index
    def update_chunk(self, index, chunk):
        self.chunks[index] = chunk

    # Delete chunk at given index
    def delete_chunk(self, index):
        self.chunks[index] = None

    # Get chunk from given index
    def get_chunk(self, index):
        return self.chunks[index]

    # Get all chunks
    def get_all_chunks(self):
        return self.chunks

    # Get number of chunks stored
    def get_num_chunks(self):
        return sum(1 if chunk is not None else 0 for chunk in self.chunks)

    # Get free space
    def get_free_space(self):
        return len(self.chunks) - self.get_num_chunks()

### NameNode

In [4]:
@ray.remote
class NameNode:
    def __init__(self, data_nodes, num_copies, max_chunk_size):
        self.data_nodes = data_nodes
        self.num_copies = num_copies
        self.max_chunk_size = max_chunk_size
        self.artifacts = {} # name -> list of lists of chunk locations (i.e. tuple (data_node_index, chunk_index))
        self.node_statuses = [True for _ in range(len(data_nodes))]

    # Gets total free space
    def get_total_free_space(self):
        calls = [node.get_free_space.remote() for i, node in enumerate(self.data_nodes) if self.node_statuses[i]]
        return sum(ray.get(calls))

    # Gets data nodes indices sorted by free space descending
    def get_sorted_nodes_indices(self):
        calls_with_nodes = [(node.get_free_space.remote(), i) for i, node in enumerate(self.data_nodes) if self.node_statuses[i]]
        free_spaces_with_nodes = [(ray.get(call[0]), call[1]) for call in calls_with_nodes]
        return [fs_and_node[1] for fs_and_node in sorted(free_spaces_with_nodes, reverse=True)]

    # Upload a single chunk
    def upload_chunk(self, artifact_name, chunk):
        node_indices = self.get_sorted_nodes_indices()[:self.num_copies]
        calls = [self.data_nodes[index].add_chunk.remote(chunk) for index in node_indices]
        chunk_indices = ray.get(calls)
        self.artifacts[artifact_name].append(list(zip(node_indices, chunk_indices)))

    # Uploads an artifcat
    def upload_artifact(self, name, data):
        if name in self.artifacts.keys():
            print("ERROR: Artifact exists")
            return

        num_chunks = int(np.ceil(len(data) / self.max_chunk_size))
        if num_chunks * self.num_copies > self.get_total_free_space():
            print("ERROR: Not enough free space for artifact")
            return

        self.artifacts[name] = list()
        pos = 0
        while pos <= len(data):
            chunk = data[pos:pos+self.max_chunk_size]
            self.upload_chunk(name, chunk)
            pos += self.max_chunk_size

    # Gets single chunk of artifact
    def get_single_chunk(self, name, chunk_num):
        for node_index, chunk_index in self.artifacts[name][chunk_num]:
            if not self.node_statuses[node_index]:
                continue
            chunk = ray.get(self.data_nodes[node_index].get_chunk.remote(chunk_index))
            if chunk is not None:
                return chunk
        return None
        
    # Update artifact
    def update_artifact(self, name, data):
        if name not in self.artifacts.keys():
            print("ERROR: Artifact doesnt exist")
            return

        pos = 0
        num_chunks = 0
        while pos < len(data):
            num_chunks += 1
            chunk = data[pos:pos+self.max_chunk_size]
            # new data has more chunks than the old one, just upload new chunks
            if num_chunks > len(self.artifacts[name]):
                self.upload_chunk(name, chunk)
            # update old chunk if it's different
            else:
                old_chunk = self.get_single_chunk(name, num_chunks-1)
                if chunk != old_chunk:
                    for node_index, chunk_index in self.artifacts[name][num_chunks-1]:
                        if not self.node_statuses[node_index]:
                            continue
                        self.data_nodes[node_index].update_chunk.remote(chunk_index, chunk)
            pos += self.max_chunk_size

        # if chunk number is smaller than before, remove rest of chunks
        for chunk_locations in self.artifacts[name][num_chunks:]:
            for node_index, chunk_index in chunk_locations:
                if not self.node_statuses[node_index]:
                    continue
                self.data_nodes[node_index].delete_chunk.remote(chunk_index)
        self.artifacts[name] = self.artifacts[name][:num_chunks]

    # Delete artifact
    def delete_artifact(self, name):
        if name not in self.artifacts.keys():
            print("ERROR: Artifact doesnt exist")
            return
        
        for chunk_locations in self.artifacts[name]:
            for node_index, chunk_index in chunk_locations:
                if not self.node_statuses[node_index]:
                    continue
                self.data_nodes[node_index].delete_chunk.remote(chunk_index)

        self.artifacts.pop(name)

    # Get artifact
    def get_artifact(self, name):
        if name not in self.artifacts.keys():
            print("ERROR: Artifact doesnt exist")
            return
            
        first_copies = [locations[0] for locations in self.artifacts[name]]
        calls = [
            self.data_nodes[node_index].get_chunk.remote(chunk_index) if self.node_statuses[node_index] else None
            for node_index, chunk_index in first_copies
        ]
        chunks = [ray.get(call) if call is not None else None for call in calls]
        
        # Try copies if corrupted
        for copy in range(1, self.num_copies):
            corrupted_indices = list()
            for chunk_num, chunk in enumerate(chunks):
                if chunk is None:
                    corrupted_indices.append(chunk_num)
            copies = [self.artifacts[name][chunk_num][copy] for chunk_num in corrupted_indices]
            calls = [
                self.data_nodes[node_index].get_chunk.remote(chunk_index) if self.node_statuses[node_index] else None
                for node_index, chunk_index in copies
            ]
            for i, index in enumerate(corrupted_indices):
                chunks[index] = ray.get(calls[i]) if calls[i] else None

        # If no copy available - inform that chunk corrupted
        for chunk_num, chunk in enumerate(chunks):
            if chunk is None:
                chunks[chunk_num] = "[CORRUPTED]"
        return ''.join(chunks)

    # Kill node
    def kill_node(self, node_index):
        self.data_nodes[node_index] = None
        self.node_statuses[node_index] = False

    # Print info
    def print_info(self):
        # Print data nodes info
        calls = [None for _ in range(len(self.data_nodes))]
        for index, node in enumerate(self.data_nodes):
            if not self.node_statuses[index]:
                continue
            calls[index] = node.get_all_chunks.remote()
        print("Data nodes:")
        for num, call in enumerate(calls):
            print(f"{num}: ", end="")
            if call is None:
                print("Dead")
            else:
                for chunk in ray.get(call):
                    print(chunk if chunk is not None else "-", end=" | ")
                print()

        # Print artifact info
        print("\nArtifacts:")
        for name in self.artifacts.keys():
            print(f"{name} - {self.get_artifact(name)}")

        # Print chunk locations info
        print("\nChunk locations:")
        for name in self.artifacts.keys():
            print(f"{name}:")
            for chunk_num, locations in enumerate(self.artifacts[name]):
                print(f"\tChunk {chunk_num+1}:")
                for node_index, chunk_index in locations:
                    print(f"\t\tData node {node_index}", end="")
                    print(f", chunk index {chunk_index}" if self.node_statuses[node_index] else " - dead")

### Tests

In [5]:
MAX_CHUNK_SIZE = 5
CHUNKS_PER_NODE = 7
NUM_COPIES = 3
NUM_DATA_NODES = 4

In [6]:
data_nodes = [DataNode.remote(CHUNKS_PER_NODE) for _ in range(NUM_DATA_NODES)]
name_node = NameNode.remote(data_nodes, NUM_COPIES, MAX_CHUNK_SIZE)

Uploading artifacts

In [7]:
ray.get(name_node.upload_artifact.remote("art1", "bardzo długi tekst"))
ray.get(name_node.upload_artifact.remote("art2", "krótszy tekst"))
ray.get(name_node.print_info.remote())

(NameNode pid=932) Data nodes:
(NameNode pid=932) 0: o dłu | gi te | kst | zy te | kst | - | - | 
(NameNode pid=932) 1: bardz | gi te | kst | króts | kst | - | - | 
(NameNode pid=932) 2: bardz | o dłu | kst | króts | zy te | - | - | 
(NameNode pid=932) 3: bardz | o dłu | gi te | króts | zy te | kst | - | 
(NameNode pid=932) 
(NameNode pid=932) Artifacts:
(NameNode pid=932) art1 - bardzo długi tekst
(NameNode pid=932) art2 - krótszy tekst
(NameNode pid=932) 
(NameNode pid=932) Chunk locations:
(NameNode pid=932) art1:
(NameNode pid=932) 	Chunk 1:
(NameNode pid=932) 		Data node 3, chunk index 0
(NameNode pid=932) 		Data node 2, chunk index 0
(NameNode pid=932) 		Data node 1, chunk index 0
(NameNode pid=932) 	Chunk 2:
(NameNode pid=932) 		Data node 0, chunk index 0
(NameNode pid=932) 		Data node 3, chunk index 1
(NameNode pid=932) 		Data node 2, chunk index 1
(NameNode pid=932) 	Chunk 3:
(NameNode pid=932) 		Data node 1, chunk index 1
(NameNode pid=932) 		Data node 0, chunk index 1
(NameN

Updating artifacts

In [8]:
ray.get(name_node.update_artifact.remote("art1", "bardzo-długi"))
ray.get(name_node.print_info.remote())

(NameNode pid=932) Data nodes:
(NameNode pid=932) 0: o-dłu | gi | - | zy te | kst | - | - | 
(NameNode pid=932) 1: bardz | gi | - | króts | kst | - | - | 
(NameNode pid=932) 2: bardz | o-dłu | - | króts | zy te | - | - | 
(NameNode pid=932) 3: bardz | o-dłu | gi | króts | zy te | kst | - | 
(NameNode pid=932) 
(NameNode pid=932) Artifacts:
(NameNode pid=932) art1 - bardzo-długi
(NameNode pid=932) art2 - krótszy tekst
(NameNode pid=932) 
(NameNode pid=932) Chunk locations:
(NameNode pid=932) art1:
(NameNode pid=932) 	Chunk 1:
(NameNode pid=932) 		Data node 3, chunk index 0
(NameNode pid=932) 		Data node 2, chunk index 0
(NameNode pid=932) 		Data node 1, chunk index 0
(NameNode pid=932) 	Chunk 2:
(NameNode pid=932) 		Data node 0, chunk index 0
(NameNode pid=932) 		Data node 3, chunk index 1
(NameNode pid=932) 		Data node 2, chunk index 1
(NameNode pid=932) 	Chunk 3:
(NameNode pid=932) 		Data node 1, chunk index 1
(NameNode pid=932) 		Data node 0, chunk index 1
(NameNode pid=932) 		Data n

Deleting artifacts

In [9]:
ray.get(name_node.delete_artifact.remote("art2"))
ray.get(name_node.print_info.remote())

(NameNode pid=932) Data nodes:
(NameNode pid=932) 0: o-dłu | gi | - | - | - | - | - | 
(NameNode pid=932) 1: bardz | gi | - | - | - | - | - | 
(NameNode pid=932) 2: bardz | o-dłu | - | - | - | - | - | 
(NameNode pid=932) 3: bardz | o-dłu | gi | - | - | - | - | 
(NameNode pid=932) 
(NameNode pid=932) Artifacts:
(NameNode pid=932) art1 - bardzo-długi
(NameNode pid=932) 
(NameNode pid=932) Chunk locations:
(NameNode pid=932) art1:
(NameNode pid=932) 	Chunk 1:
(NameNode pid=932) 		Data node 3, chunk index 0
(NameNode pid=932) 		Data node 2, chunk index 0
(NameNode pid=932) 		Data node 1, chunk index 0
(NameNode pid=932) 	Chunk 2:
(NameNode pid=932) 		Data node 0, chunk index 0
(NameNode pid=932) 		Data node 3, chunk index 1
(NameNode pid=932) 		Data node 2, chunk index 1
(NameNode pid=932) 	Chunk 3:
(NameNode pid=932) 		Data node 1, chunk index 1
(NameNode pid=932) 		Data node 0, chunk index 1
(NameNode pid=932) 		Data node 3, chunk index 2


Getting artifact

In [10]:
print(f"art1: {ray.get(name_node.get_artifact.remote('art1'))}")

art1: bardzo-długi


Killing node

In [11]:
ray.get(name_node.kill_node.remote(2))
ray.get(name_node.print_info.remote())

(NameNode pid=932) Data nodes:
(NameNode pid=932) 0: o-dłu | gi | - | - | - | - | - | 
(NameNode pid=932) 1: bardz | gi | - | - | - | - | - | 
(NameNode pid=932) 2: Dead
(NameNode pid=932) 3: bardz | o-dłu | gi | - | - | - | - | 
(NameNode pid=932) 
(NameNode pid=932) Artifacts:
(NameNode pid=932) art1 - bardzo-długi
(NameNode pid=932) 
(NameNode pid=932) Chunk locations:
(NameNode pid=932) art1:
(NameNode pid=932) 	Chunk 1:
(NameNode pid=932) 		Data node 3, chunk index 0
(NameNode pid=932) 		Data node 2 - dead
(NameNode pid=932) 		Data node 1, chunk index 0
(NameNode pid=932) 	Chunk 2:
(NameNode pid=932) 		Data node 0, chunk index 0
(NameNode pid=932) 		Data node 3, chunk index 1
(NameNode pid=932) 		Data node 2 - dead
(NameNode pid=932) 	Chunk 3:
(NameNode pid=932) 		Data node 1, chunk index 1
(NameNode pid=932) 		Data node 0, chunk index 1
(NameNode pid=932) 		Data node 3, chunk index 2


Killing too much nodes

In [12]:
ray.get(name_node.kill_node.remote(1))
ray.get(name_node.kill_node.remote(3))
ray.get(name_node.print_info.remote())

(NameNode pid=932) Data nodes:
(NameNode pid=932) 0: o-dłu | gi | - | - | - | - | - | 
(NameNode pid=932) 1: Dead
(NameNode pid=932) 2: Dead
(NameNode pid=932) 3: Dead
(NameNode pid=932) 
(NameNode pid=932) Artifacts:
(NameNode pid=932) art1 - [CORRUPTED]o-długi
(NameNode pid=932) 
(NameNode pid=932) Chunk locations:
(NameNode pid=932) art1:
(NameNode pid=932) 	Chunk 1:
(NameNode pid=932) 		Data node 3 - dead
(NameNode pid=932) 		Data node 2 - dead
(NameNode pid=932) 		Data node 1 - dead
(NameNode pid=932) 	Chunk 2:
(NameNode pid=932) 		Data node 0, chunk index 0
(NameNode pid=932) 		Data node 3 - dead
(NameNode pid=932) 		Data node 2 - dead
(NameNode pid=932) 	Chunk 3:
(NameNode pid=932) 		Data node 1 - dead
(NameNode pid=932) 		Data node 0, chunk index 1
(NameNode pid=932) 		Data node 3 - dead
